In [0]:
%sql
select * from list_secrets()

In [0]:
import requests
import json
import logging
import base64
from datetime import datetime, timezone
from jinja2 import Template

# ==========================================
# 1. Configuration & Setup
# ==========================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

try:
    NVIDIA_API_KEY = dbutils.secrets.get("courseify", "nvidia-api-key-1")
    GITHUB_TOKEN = dbutils.secrets.get("courseify", " github-token-daily-commit")
    # GITHUB_TOKEN = ""
except Exception as e:
    logger.error("Failed to fetch API keys from Databricks secrets.")
    raise e

TABLE_NAME = "courseify.default.jobs_bronze"
REPO_OWNER = "courseify-jobs-1"  # <--- MARCHU (e.g., sumanth123)
REPO_NAME = "courseify_jobs_portal"         # <--- MARCHU (e.g., courseify-web)
BRANCH = "main"                      

# ==========================================
# 2. Generate Google-News Ready SEO Blog
# ==========================================
def generate_daily_seo_blog():
    """Generates a high-quality tech blog using NVIDIA Llama 3.1 for AdSense/Google News."""
    logger.info("Generating daily SEO blog using NVIDIA LLM...")
    invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_API_KEY}", 
        "Content-Type": "application/json"
    }
    
    # Prompt is designed to generate Google News worthy content
    prompt = """
    You are an expert Tech Journalist writing for a Google News approved tech blog in India.
    Write a short, highly engaging, SEO-optimized tech career update for today.
    Topics: AI trends, Databricks, PySpark, Full-stack dev, or IT hiring trends in India.
    Make the title catchy. The content should be 3-4 sentences of solid, actionable tech advice.
    Return STRICTLY in JSON format: {"title": "Catchy Title", "content": "Your solid advice here"}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.6, 
        "max_tokens": 512
    }
    
    try:
        response = requests.post(invoke_url, headers=headers, json=payload)
        response.raise_for_status()
        result_text = response.json()['choices'][0]['message']['content']
        clean_json_str = result_text.replace("```json", "").replace("```", "").strip()
        blog_data = json.loads(clean_json_str)
        return blog_data
    except Exception as e:
        logger.error(f"Failed to generate blog: {e}")
        # Fallback content if API fails
        return {
            "title": "How to Build a Zero-Cost Data Engineering Pipeline in 2026", 
            "content": "Learn how combining Databricks Community Edition with GitHub Pages can create a robust, fully automated serverless architecture without spending a single rupee. Perfect for scaling your portfolio."
        }

# ==========================================
# 3. Fetch Jobs & Push to GitHub Pages
# ==========================================
def update_github_pages():
    logger.info("Starting Website Publisher Process...")
    
    # 1. Fetch Latest 30 Active Jobs from Databricks Table
    try:
        # Sort by inserted_at to always get the newest jobs
        latest_jobs_df = spark.sql(f"""
            SELECT title, company, location, type, salary, category, apply_link 
            FROM {TABLE_NAME}
            ORDER BY inserted_at DESC
            LIMIT 30
        """)
        jobs_data = [row.asDict() for row in latest_jobs_df.collect()]
        logger.info(f"✅ Fetched {len(jobs_data)} latest jobs from {TABLE_NAME}.")
    except Exception as e:
        logger.error(f"Failed to fetch jobs from table: {e}")
        return

    # 2. Generate Today's SEO Blog
    daily_blog = generate_daily_seo_blog()
    logger.info(f"✅ Generated Blog: {daily_blog.get('title')}")

    # 3. Fetch the raw Jinja template file from your GitHub repository
    template_url = f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{BRANCH}/index_template.html"
    headers = {"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"}
    
    template_resp = requests.get(template_url, headers=headers)
    if template_resp.status_code != 200:
        logger.error(f"Failed to fetch index_template.html from GitHub. Check REPO_OWNER/REPO_NAME.")
        return
        
    # 4. Render HTML using Jinja2
    template = Template(template_resp.text)
    final_html = template.render(
        jobs=jobs_data, 
        daily_blog=daily_blog, 
        total_jobs=len(jobs_data)
    )
    
    # 5. Push the final index.html to GitHub
    file_path = "index.html"
    api_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/{file_path}"
    
    # Need the current file's SHA hash to overwrite it in GitHub
    existing_file_resp = requests.get(api_url, headers=headers)
    file_sha = existing_file_resp.json().get("sha", "") if existing_file_resp.status_code == 200 else ""
    
    push_data = {
        "message": f"Auto-Deploy: Updated {len(jobs_data)} Jobs & Daily Blog",
        "content": base64.b64encode(final_html.encode("utf-8")).decode("utf-8"),
        "branch": BRANCH
    }
    
    if file_sha:
        push_data["sha"] = file_sha  # Required to update an existing file
        
    push_resp = requests.put(api_url, headers=headers, json=push_data)
    
    if push_resp.status_code in [200, 201]:
        logger.info("🚀 Successfully pushed index.html to GitHub! Your site will be updated in ~1 minute.")
    else:
        logger.error(f"❌ Failed to push to GitHub: {push_resp.json()}")

# ==========================================
# 4. Main Execution
# ==========================================
if __name__ == "__main__":
    update_github_pages()